# Misinformation Detection Pipeline — Full Version

### Changes in this version
- **NLI-typed edges**: `cross-encoder/nli-deberta-v3-small` classifies each edge as entailment / contradiction / neutral. Node features now include contradiction counts — a claim contradicted by high-scoring evidence is a strong fake signal.
- **DBSCAN consensus clustering**: replaces average-similarity document ranking. The largest cluster = consensus. Outlier documents get penalized rather than just averaged.
- **GAT with residual connections + JumpingKnowledge**: 4-layer Graph Attention Network. Residual connections prevent over-smoothing. JumpingKnowledge concatenates all layer outputs before pooling so early-layer structure isn't lost. Multi-pooling (mean+max) for richer graph-level representation.
- **Expanded node features (14)**: adds NLI-based contradiction/entailment counts per node.
- **DB fix**: confidence threshold lowered, updates from all data not just test, connection handling fixed.

In [1]:
# pip install spacy duckduckgo-search
# python -m spacy download en_core_web_sm

import nltk
try:
    nltk.download('punkt_tab', quiet=True)
except Exception:
    nltk.download('punkt', quiet=True)

import spacy
nlp = spacy.load('en_core_web_sm')

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

ENCODER = SentenceTransformer('all-MiniLM-L6-v2')   # sentence embeddings
NLI_MODEL = CrossEncoder('cross-encoder/nli-deberta-v3-small')  # entailment classifier
# NLI label order for this model: contradiction=0, entailment=1, neutral=2
NLI_CONTRADICTION = 0
NLI_ENTAILMENT    = 1
NLI_NEUTRAL       = 2

EMBEDDING_DIM = 384
FEATURE_DIM   = 14   # expanded from 8

print('Setup complete.')

config.json: 0.00B [00:00, ?B/s]

c:\Users\bhada\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bhada\.cache\huggingface\hub\models--cross-encoder--nli-deberta-v3-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Setup complete.


## Step 1: Fact Extraction

In [2]:
import networkx as nx
from nltk.tokenize import sent_tokenize

def extract_facts(text: str) -> list[tuple[str, np.ndarray]]:
    sentences = sent_tokenize(text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    if not sentences:
        return []
    embeddings = ENCODER.encode(sentences, batch_size=32, show_progress_bar=False)
    return list(zip(sentences, embeddings))

## Step 2: NLI-Typed Knowledge Graph

Edges are now typed: **entailment** (one claim supports another), **contradiction** (one claim opposes another), or **neutral** (topically related but neither supports nor contradicts). This distinction is what BERT embeddings alone cannot make — two sentences can be semantically close in embedding space but logically opposite.

The NLI model runs on sentence pairs that already pass the cosine similarity threshold, so we only call it on pairs that are at least topically related — keeping it fast.

In [3]:
# Edge type constants stored as integers on graph edges
EDGE_NEUTRAL       = 0
EDGE_ENTAILMENT    = 1
EDGE_CONTRADICTION = 2


def classify_edge_nli(sent_a: str, sent_b: str) -> tuple[int, float]:
    """
    Run NLI model on a sentence pair.
    Returns (edge_type, confidence) where confidence is the softmax prob
    of the winning class.

    The CrossEncoder returns raw logits for [contradiction, entailment, neutral].
    We softmax to get probabilities, then take argmax.
    """
    logits = NLI_MODEL.predict([(sent_a, sent_b)])[0]  # shape (3,)
    probs  = np.exp(logits) / np.exp(logits).sum()      # softmax
    label  = int(np.argmax(probs))
    conf   = float(probs[label])
    # Map NLI labels to our edge type constants
    if label == NLI_CONTRADICTION:
        return EDGE_CONTRADICTION, conf
    elif label == NLI_ENTAILMENT:
        return EDGE_ENTAILMENT, conf
    else:
        return EDGE_NEUTRAL, conf


def build_knowledge_graph(facts: list[tuple[str, np.ndarray]],
                           sim_threshold: float = 0.60,
                           use_nli: bool = True) -> nx.Graph:
    """
    Build graph with NLI-typed edges.

    sim_threshold lowered from 0.75 → 0.60: NLI now handles the quality
    filter, so we cast a wider net on cosine similarity to give NLI more
    pairs to classify. Pairs below 0.60 are genuinely unrelated.

    use_nli=False skips NLI classification (faster, for debugging).
    """
    G = nx.Graph()
    if not facts:
        return G

    sentences, embeddings = zip(*facts)
    embeddings = np.array(embeddings)

    for i, (sent, emb) in enumerate(zip(sentences, embeddings)):
        G.add_node(i, text=sent, embedding=emb, source='input', weight=1.0)

    sim_matrix = cosine_similarity(embeddings)
    candidate_pairs = [
        (i, j) for i in range(len(sentences))
        for j in range(i + 1, len(sentences))
        if sim_matrix[i, j] > sim_threshold
    ]

    if use_nli and candidate_pairs:
        # Batch NLI inference — much faster than calling one pair at a time
        pairs_text = [(sentences[i], sentences[j]) for i, j in candidate_pairs]
        all_logits = NLI_MODEL.predict(pairs_text)  # shape (n_pairs, 3)
        all_probs  = np.exp(all_logits) / np.exp(all_logits).sum(axis=1, keepdims=True)
        labels     = np.argmax(all_probs, axis=1)
        confs      = all_probs[np.arange(len(labels)), labels]

        for (i, j), label, conf in zip(candidate_pairs, labels, confs):
            edge_type = EDGE_CONTRADICTION if label == NLI_CONTRADICTION else \
                        EDGE_ENTAILMENT    if label == NLI_ENTAILMENT    else \
                        EDGE_NEUTRAL
            G.add_edge(i, j,
                       similarity=float(sim_matrix[i, j]),
                       edge_type=edge_type,
                       nli_confidence=float(conf))
    else:
        for i, j in candidate_pairs:
            G.add_edge(i, j, similarity=float(sim_matrix[i, j]),
                       edge_type=EDGE_NEUTRAL, nli_confidence=0.5)

    return G

## Step 3: Atomic Claim Extraction

In [4]:
VERIFIABLE_ENTITY_TYPES = {'PERSON', 'ORG', 'GPE', 'DATE', 'CARDINAL',
                            'NORP', 'FAC', 'PRODUCT', 'EVENT', 'LAW'}

def extract_svo(sent) -> str | None:
    subject = verb = obj = None
    for token in sent:
        if token.dep_ in ('nsubj', 'nsubjpass') and subject is None:
            subject = ' '.join(t.text for t in token.subtree
                               if t.dep_ in ('compound', 'amod', 'nsubj', 'nsubjpass') or t == token)
        if token.pos_ == 'VERB' and token.dep_ in ('ROOT', 'relcl') and verb is None:
            verb = token.lemma_
        if token.dep_ in ('dobj', 'attr', 'pobj') and obj is None:
            obj = ' '.join(t.text for t in token.subtree
                           if t.dep_ in ('compound', 'amod', 'dobj', 'attr', 'pobj') or t == token)
    parts = [p for p in [subject, verb, obj] if p]
    return ' '.join(parts) if len(parts) >= 2 else None


def extract_atomic_claims(text: str, max_claims: int = 3) -> list[str]:
    doc = nlp(text[:5000])
    scored = []
    for sent in doc.sents:
        if len(sent.text.strip()) < 20:
            continue
        verifiable = {ent.label_ for ent in sent.ents} & VERIFIABLE_ENTITY_TYPES
        if verifiable:
            scored.append((len(verifiable), sent))
    scored.sort(key=lambda x: x[0], reverse=True)
    claims = []
    for _, sent in scored[:max_claims * 2]:
        if len(claims) >= max_claims:
            break
        query = extract_svo(sent) or sent.text.strip()[:100]
        if query:
            claims.append(query)
    return claims

## Step 4: Credibility Database (Fixed)

**Bug fix:** The previous version used `confidence > 0.3` as a gate before updating the DB. Since the model was outputting ~0.5 for everything (stuck at random), confidence was always ~0 and nothing was ever written. The fix is to lower the threshold to 0.1 — a mild preference is still a signal — and to also update from the full training/val set, not just test.

In [5]:
import sqlite3
import math
from contextlib import contextmanager
from urllib.parse import urlparse
from datetime import datetime

DB_PATH      = 'source_credibility.db'
MODEL_WEIGHT = 0.3
USER_WEIGHT  = 1.0


@contextmanager
def get_db():
    """
    Context manager for DB connections — guarantees the connection is
    closed even if an exception occurs. The previous version returned a
    bare connection which caused ResourceWarnings about unclosed handles.
    """
    conn = sqlite3.connect(DB_PATH)
    try:
        yield conn
        conn.commit()
    finally:
        conn.close()


def init_db():
    with get_db() as conn:
        conn.execute('''
            CREATE TABLE IF NOT EXISTS sources (
                domain        TEXT PRIMARY KEY,
                alpha         REAL DEFAULT 2.0,
                beta          REAL DEFAULT 2.0,
                model_updates INTEGER DEFAULT 0,
                user_updates  INTEGER DEFAULT 0,
                last_updated  TEXT
            )''')
        conn.execute('''
            CREATE TABLE IF NOT EXISTS credibility_log (
                id          INTEGER PRIMARY KEY AUTOINCREMENT,
                domain      TEXT,
                signal_type TEXT,
                signal      TEXT,
                confidence  REAL,
                timestamp   TEXT
            )''')

init_db()


def extract_domain(url: str) -> str:
    return urlparse(url).netloc.replace('www.', '')


def get_credibility(domain: str) -> float:
    with get_db() as conn:
        row = conn.execute(
            'SELECT alpha, beta FROM sources WHERE domain = ?', (domain,)
        ).fetchone()
    return (row[0] / (row[0] + row[1])) if row else 0.5


def get_credibility_report(domain: str) -> dict:
    with get_db() as conn:
        row = conn.execute(
            'SELECT alpha, beta, model_updates, user_updates FROM sources WHERE domain = ?',
            (domain,)
        ).fetchone()
    if row is None:
        return {'domain': domain, 'score': 0.5, 'n_signals': 0, 'confidence': 'none'}
    alpha, beta, m, u = row
    score = alpha / (alpha + beta)
    var   = (alpha * beta) / ((alpha + beta) ** 2 * (alpha + beta + 1))
    return {
        'domain': domain, 'score': round(score, 3),
        'std': round(math.sqrt(var), 3), 'n_signals': int(alpha + beta - 4),
        'model_updates': m, 'user_updates': u,
        'confidence': 'high' if (alpha+beta-4) > 20 else 'medium' if (alpha+beta-4) > 5 else 'low'
    }


def update_credibility(domain: str, signal: str, signal_type: str, confidence: float = 1.0):
    assert signal in ('real', 'fake') and signal_type in ('model', 'user')
    weight = (MODEL_WEIGHT if signal_type == 'model' else USER_WEIGHT) * confidence
    now    = datetime.utcnow().isoformat()
    with get_db() as conn:
        conn.execute(
            'INSERT OR IGNORE INTO sources (domain, alpha, beta, last_updated) VALUES (?,2.0,2.0,?)',
            (domain, now)
        )
        col        = 'alpha' if signal == 'real' else 'beta'
        update_col = 'model_updates' if signal_type == 'model' else 'user_updates'
        conn.execute(
            f'UPDATE sources SET {col}={col}+?, {update_col}={update_col}+1, last_updated=? WHERE domain=?',
            (weight, now, domain)
        )
        conn.execute(
            'INSERT INTO credibility_log (domain,signal_type,signal,confidence,timestamp) VALUES(?,?,?,?,?)',
            (domain, signal_type, signal, confidence, now)
        )


def bulk_update_from_prediction(scored_docs: list[tuple[dict, float]],
                                 prediction: float, model_confidence: float,
                                 confidence_threshold: float = 0.1):
    """
    Update credibility DB for all evidence sources after a model prediction.
    Threshold lowered from 0.3 → 0.1 — even a mild model preference is a signal.
    confidence_threshold=0.0 would update on every single prediction.
    """
    if model_confidence < confidence_threshold:
        return 0
    signal = 'fake' if prediction > 0.5 else 'real'
    n = 0
    for doc, doc_score in scored_docs:
        url = doc.get('link', '')
        if not url:
            continue
        domain = extract_domain(url)
        update_credibility(domain, signal, 'model',
                           confidence=model_confidence * doc_score)
        n += 1
    return n


print('Credibility DB initialised at', DB_PATH)

Credibility DB initialised at source_credibility.db


## Step 5: DuckDuckGo Search with Disk Cache

In [6]:
import json
import os
import hashlib
import time
from duckduckgo_search import DDGS

CACHE_DIR = 'search_cache'
os.makedirs(CACHE_DIR, exist_ok=True)


def _cache_path(query: str) -> str:
    return os.path.join(CACHE_DIR, hashlib.md5(query.encode()).hexdigest() + '.json')


def ddg_search(query: str, num_results: int = 10) -> list[dict]:
    cache_file = _cache_path(query)
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            return json.load(f)
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=num_results):
            results.append({
                'link':    r.get('href', ''),
                'title':   r.get('title', ''),
                'snippet': r.get('body', '')
            })
    with open(cache_file, 'w') as f:
        json.dump(results, f)
    time.sleep(0.5)  # gentle rate limiting — cache means this only hits once per query
    return results


def collect_evidence(text: str, k: int = 10) -> list[dict]:
    claims = extract_atomic_claims(text, max_claims=3)
    if not claims:
        return []
    seen_links, all_results = set(), []
    for claim in claims:
        try:
            results = ddg_search(claim, num_results=k)
        except Exception as e:
            print(f'  Search failed for "{claim[:60]}": {e}')
            continue
        for r in results:
            if r.get('link') not in seen_links:
                seen_links.add(r['link'])
                all_results.append(r)
    return all_results

## Step 6: DBSCAN Consensus Clustering for Document Scoring

**Why DBSCAN instead of average similarity?** Average similarity treats all documents symmetrically — an outlier document that happens to be somewhat similar to several others can score deceptively high. DBSCAN finds the **largest dense cluster** of documents (the consensus) and scores each document by whether it belongs to that cluster and how central it is. Documents in small clusters or labeled as noise get penalized.

The analogy: if 8 sources agree and 2 say something completely different, average similarity would still give those 2 a moderate score. DBSCAN labels them as outliers and scores them near zero.

In [7]:
from sklearn.cluster import DBSCAN


def score_documents_dbscan(documents: list[dict],
                            credibility_alpha: float = 0.6,
                            eps: float = 0.3,
                            min_samples: int = 2) -> list[tuple[dict, float]]:
    """
    Rank documents by DBSCAN consensus cluster membership + domain credibility.

    eps: DBSCAN neighbourhood radius in embedding space (0.3 ≈ cosine dist 0.7 similarity)
    min_samples: minimum cluster size — lone documents are noise.

    Scoring:
      - Documents in the largest cluster: score based on centrality within cluster
      - Documents in smaller clusters:    score halved
      - Noise (label = -1):               score = 0.1 (don't completely ignore, just downweight)
    """
    snippets = [doc.get('snippet', '') for doc in documents]
    if not snippets:
        return []

    embeddings = ENCODER.encode(snippets, batch_size=32, show_progress_bar=False)
    # Convert cosine similarity to distance (DBSCAN expects distance, not similarity)
    sim_matrix  = cosine_similarity(embeddings)
    dist_matrix = np.clip(1.0 - sim_matrix, 0, 2).astype(np.float64)

    clustering = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed')
    labels     = clustering.fit_predict(dist_matrix)

    # Find the largest cluster (most sources agree)
    unique_labels = [l for l in set(labels) if l != -1]
    if unique_labels:
        cluster_sizes  = {l: (labels == l).sum() for l in unique_labels}
        consensus_label = max(cluster_sizes, key=cluster_sizes.get)
    else:
        consensus_label = None  # all noise — fall back to average similarity

    scored = []
    for i, doc in enumerate(documents):
        label = labels[i]

        if consensus_label is not None:
            if label == consensus_label:
                # Centrality within cluster = average similarity to other cluster members
                cluster_members = np.where(labels == consensus_label)[0]
                cluster_sims    = sim_matrix[i, cluster_members]
                consensus_score = float(cluster_sims.mean())
            elif label == -1:
                consensus_score = 0.1  # noise: still include but highly downweighted
            else:
                consensus_score = float(sim_matrix[i].mean()) * 0.5  # minority cluster
        else:
            # No clusters found: fall back to average similarity
            consensus_score = float((sim_matrix[i].sum() - 1.0) / max(len(documents) - 1, 1))

        # Blend with stored domain credibility
        domain      = extract_domain(doc.get('link', ''))
        credibility = get_credibility(domain)
        combined    = credibility_alpha * consensus_score + (1 - credibility_alpha) * credibility
        scored.append((doc, float(combined)))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

## Step 7: Graph Augmentation with NLI-Typed Evidence Edges

In [8]:
def augment_knowledge_graph(G: nx.Graph,
                             scored_docs: list[tuple[dict, float]],
                             sim_threshold: float = 0.60,
                             score_threshold: float = 0.3,
                             use_nli: bool = True) -> nx.Graph:
    """
    Add evidence nodes and NLI-typed edges from ranked documents.
    Evidence↔input edges are also NLI-classified so contradicting evidence
    is structurally distinct from supporting evidence in the graph.
    """
    existing_nodes = list(G.nodes(data=True))
    next_id        = max(G.nodes()) + 1 if G.nodes() else 0
    new_nodes, new_edges = [], []

    for doc, doc_score in scored_docs:
        if doc_score < score_threshold:
            continue
        snippet = doc.get('snippet', '')
        if not snippet:
            continue
        evidence_facts = extract_facts(snippet)

        for fact_text, fact_emb in evidence_facts:
            nid = next_id
            next_id += 1
            new_nodes.append((nid, fact_text, fact_emb, doc_score))

            fact_emb_2d = fact_emb.reshape(1, -1)
            for existing_id, data in existing_nodes:
                sim = float(cosine_similarity(fact_emb_2d, data['embedding'].reshape(1, -1))[0, 0])
                if sim > sim_threshold:
                    if use_nli:
                        edge_type, nli_conf = classify_edge_nli(fact_text, data['text'])
                    else:
                        edge_type, nli_conf = EDGE_NEUTRAL, 0.5
                    new_edges.append((nid, existing_id, sim, doc_score, edge_type, nli_conf))

    for nid, text, emb, score in new_nodes:
        G.add_node(nid, text=text, embedding=emb, source='evidence', weight=score)
    for nid, eid, sim, score, etype, nli_conf in new_edges:
        G.add_edge(nid, eid, similarity=sim, evidence_weight=score,
                   edge_type=etype, nli_confidence=nli_conf)
    return G

## Step 8: Expanded Structural Node Features (14 dimensions)

Added 6 NLI-aware features on top of the original 8:
- `n_entailing_evidence`, `n_contradicting_evidence`: raw counts of supporting/opposing evidence
- `entailment_weight_sum`, `contradiction_weight_sum`: weighted by source reliability
- `entailment_ratio`, `contradiction_ratio`: normalized fractions

A claim with high `contradiction_weight_sum` and high-scoring sources is a strong fake signal. A claim with high `entailment_weight_sum` is well-supported.

In [9]:
def compute_node_features(G: nx.Graph) -> np.ndarray:
    """
    14 structural features per node:

    Original 8:
      0: evidence_degree
      1: input_degree
      2: avg_evidence_weight
      3: max_evidence_weight
      4: avg_edge_similarity
      5: evidence_ratio
      6: is_evidence
      7: source_weight

    NLI-based additions:
      8:  n_entailing_evidence      — count of evidence nodes with ENTAILMENT edges
      9:  n_contradicting_evidence  — count of evidence nodes with CONTRADICTION edges
      10: entailment_weight_sum     — sum of evidence_weight × nli_confidence for entailing edges
      11: contradiction_weight_sum  — same for contradicting edges
      12: entailment_ratio          — entailing / total evidence edges
      13: contradiction_ratio       — contradicting / total evidence edges
    """
    feature_list = []
    for node_id, data in G.nodes(data=True):
        neighbors        = list(G.neighbors(node_id))
        evidence_nbrs    = [n for n in neighbors if G.nodes[n].get('source') == 'evidence']
        input_nbrs       = [n for n in neighbors if G.nodes[n].get('source') == 'input']
        evidence_weights = [G.edges[node_id, n].get('evidence_weight', 0.0) for n in evidence_nbrs]
        all_sims         = [G.edges[node_id, n].get('similarity', 0.0) for n in neighbors]

        # NLI-typed breakdown
        entailing     = [(n, G.edges[node_id, n]) for n in evidence_nbrs
                         if G.edges[node_id, n].get('edge_type') == EDGE_ENTAILMENT]
        contradicting = [(n, G.edges[node_id, n]) for n in evidence_nbrs
                         if G.edges[node_id, n].get('edge_type') == EDGE_CONTRADICTION]

        ent_weight_sum  = sum(e.get('evidence_weight', 0) * e.get('nli_confidence', 0.5)
                              for _, e in entailing)
        cont_weight_sum = sum(e.get('evidence_weight', 0) * e.get('nli_confidence', 0.5)
                              for _, e in contradicting)
        n_ev = max(len(evidence_nbrs), 1)

        feature_list.append([
            len(evidence_nbrs),                                          # 0
            len(input_nbrs),                                             # 1
            np.mean(evidence_weights) if evidence_weights else 0.0,      # 2
            np.max(evidence_weights)  if evidence_weights else 0.0,      # 3
            np.mean(all_sims)         if all_sims         else 0.0,      # 4
            len(evidence_nbrs) / max(len(neighbors), 1),                 # 5
            1.0 if data.get('source') == 'evidence' else 0.0,            # 6
            data.get('weight', 1.0),                                     # 7
            float(len(entailing)),                                       # 8
            float(len(contradicting)),                                   # 9
            ent_weight_sum,                                              # 10
            cont_weight_sum,                                             # 11
            len(entailing)    / n_ev,                                    # 12
            len(contradicting) / n_ev,                                   # 13
        ])

    arr     = np.array(feature_list, dtype=np.float32)
    col_max = arr.max(axis=0)
    col_max[col_max == 0] = 1
    return arr / col_max


def graph_to_pyg(G: nx.Graph, label: int | None = None):
    from torch_geometric.data import Data
    node_ids = list(G.nodes())
    id_map   = {nid: i for i, nid in enumerate(node_ids)}
    x        = torch.tensor(compute_node_features(G), dtype=torch.float)

    if G.edges():
        edge_list  = [(id_map[u], id_map[v]) for u, v in G.edges()]
        edge_index = torch.tensor(edge_list, dtype=torch.long).T
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        # Edge features: [edge_type_onehot(3), similarity, nli_confidence, evidence_weight]
        edge_feats = []
        for u, v in G.edges():
            ed  = G.edges[u, v]
            et  = ed.get('edge_type', EDGE_NEUTRAL)
            oh  = [1.0 if et == k else 0.0 for k in range(3)]  # one-hot
            row = oh + [ed.get('similarity', 0.0),
                        ed.get('nli_confidence', 0.5),
                        ed.get('evidence_weight', 0.0)]
            edge_feats.append(row)
        # Duplicate for reverse edges
        ef_tensor = torch.tensor(edge_feats * 2, dtype=torch.float)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        ef_tensor  = torch.zeros((0, 6), dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=ef_tensor)
    if label is not None:
        data.y = torch.tensor([label], dtype=torch.float)
    return data

## Step 9: Deep GAT with Residual Connections + JumpingKnowledge

**Why not just more GCN layers?** GCN suffers from **over-smoothing**: after k layers, every node's representation is a weighted average of its k-hop neighborhood. With enough layers, all nodes converge to the same value and the model loses all discrimination. Two countermeasures are used here:

1. **Residual connections** (like ResNet): `x = conv(x) + x` — the identity path lets gradients flow directly and prevents early-layer information being washed out.
2. **JumpingKnowledge**: concatenate the output of *every* layer before the final pooling. The model can choose to use 1-hop, 2-hop, or 4-hop neighborhood information per node independently.

**Why GAT over GCN?** GCN averages all neighbor contributions equally. GAT learns attention weights — it can focus on the few high-scoring evidence nodes that actually matter and ignore noise. This matters a lot when the graph is heterogeneous (mix of article nodes and evidence nodes with different reliability scores).

**Architecture**: 4 GAT layers (14→64→128→128→64) with residual connections, BatchNorm, and multi-head attention. Output: concat(mean_pool, max_pool) → 3-layer MLP classifier.

In [10]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data


class FakeNewsGAT(torch.nn.Module):
    """
    4-layer Graph Attention Network with:
      - Multi-head attention (4 heads)
      - Residual connections (project input to hidden_dim on first skip)
      - BatchNorm after each layer
      - JumpingKnowledge: all layer outputs concatenated before pooling
      - Multi-pooling: mean + max pooled representations concatenated
      - 3-layer MLP classifier with dropout

    Total parameters: ~150k. Large enough to capture complex patterns,
    small enough to train on 200 graphs without severe overfitting.
    """
    def __init__(self,
                 input_dim:  int   = FEATURE_DIM,
                 hidden_dim: int   = 64,
                 n_heads:    int   = 4,
                 dropout:    float = 0.4):
        super().__init__()
        self.dropout = dropout

        # Input projection (aligns input_dim → hidden_dim for residuals)
        self.input_proj = torch.nn.Linear(input_dim, hidden_dim)

        # 4 GAT layers — concat=True means output dim = hidden_dim * n_heads
        # We then project back to hidden_dim with a linear layer after each
        self.conv1 = GATConv(hidden_dim,     hidden_dim,     heads=n_heads, concat=True,  dropout=dropout)
        self.lin1  = torch.nn.Linear(hidden_dim * n_heads, hidden_dim)
        self.bn1   = torch.nn.BatchNorm1d(hidden_dim)

        self.conv2 = GATConv(hidden_dim,     hidden_dim * 2, heads=n_heads, concat=True,  dropout=dropout)
        self.lin2  = torch.nn.Linear(hidden_dim * 2 * n_heads, hidden_dim * 2)
        self.bn2   = torch.nn.BatchNorm1d(hidden_dim * 2)

        self.conv3 = GATConv(hidden_dim * 2, hidden_dim * 2, heads=n_heads, concat=True,  dropout=dropout)
        self.lin3  = torch.nn.Linear(hidden_dim * 2 * n_heads, hidden_dim * 2)
        self.bn3   = torch.nn.BatchNorm1d(hidden_dim * 2)

        self.conv4 = GATConv(hidden_dim * 2, hidden_dim,     heads=n_heads, concat=False, dropout=dropout)
        self.bn4   = torch.nn.BatchNorm1d(hidden_dim)

        # JumpingKnowledge concat: layer outputs have dims [64, 128, 128, 64]
        # → concatenated = 384 per node. Then we pool mean+max → 768 per graph.
        jk_dim = hidden_dim + hidden_dim*2 + hidden_dim*2 + hidden_dim  # 384
        pool_dim = jk_dim * 2  # mean + max concatenated = 768

        # 3-layer MLP classifier
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(pool_dim, 256),
            torch.nn.BatchNorm1d(256),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(256, 64),
            torch.nn.BatchNorm1d(64),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(64, 1)  # raw logit
        )

    def forward(self, x, edge_index, batch):
        # Project input to hidden_dim
        x = F.relu(self.input_proj(x))
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Layer 1 — residual: x1 = relu(BN(Lin(GAT(x)))) + x
        x1 = self.bn1(F.relu(self.lin1(self.conv1(x, edge_index)))) + x

        # Layer 2 — need to project residual to match new dim (64 → 128)
        res2 = F.relu(torch.nn.functional.pad(x1, (0, x1.shape[1])))  # zero-pad to 128
        x2   = self.bn2(F.relu(self.lin2(self.conv2(x1, edge_index)))) + res2

        # Layer 3 — same dim (128 → 128), direct residual
        x3 = self.bn3(F.relu(self.lin3(self.conv3(x2, edge_index)))) + x2

        # Layer 4 — project back down (128 → 64), project residual too
        res4 = x3[:, :x1.shape[1]]  # take first 64 dims as residual
        x4   = self.bn4(F.relu(self.conv4(x3, edge_index))) + res4

        # JumpingKnowledge: concatenate all layer outputs
        x_jk = torch.cat([x1, x2, x3, x4], dim=1)  # (n_nodes, 384)

        # Multi-pooling: mean + max, then concatenate
        x_mean = global_mean_pool(x_jk, batch)  # (n_graphs, 384)
        x_max  = global_max_pool(x_jk, batch)   # (n_graphs, 384)
        x_pool = torch.cat([x_mean, x_max], dim=1)  # (n_graphs, 768)

        return self.mlp(x_pool)  # (n_graphs, 1) — raw logit


def train_gat(train_data, val_data, epochs=150, lr=5e-4, patience=20):
    model     = FakeNewsGAT()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    # Cosine annealing decays LR smoothly — better than fixed LR for deeper models
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = torch.nn.BCEWithLogitsLoss()
    t_loader  = DataLoader(train_data, batch_size=16, shuffle=True)
    v_loader  = DataLoader(val_data,   batch_size=16)

    best_val, no_improve, best_state = float('inf'), 0, None

    for epoch in range(epochs):
        model.train()
        t_loss = 0
        for batch in t_loader:
            optimizer.zero_grad()
            out  = model(batch.x, batch.edge_index, batch.batch).squeeze()
            loss = criterion(out, batch.y.squeeze())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item()
        scheduler.step()

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for batch in v_loader:
                out    = model(batch.x, batch.edge_index, batch.batch).squeeze()
                v_loss += criterion(out, batch.y.squeeze()).item()

        avg_t = t_loss / len(t_loader)
        avg_v = v_loss / len(v_loader)
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:3d}  train={avg_t:.4f}  val={avg_v:.4f}  '
                  f'lr={scheduler.get_last_lr()[0]:.2e}')

        if avg_v < best_val:
            best_val   = avg_v
            no_improve = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return model


def evaluate_model(model, dataset):
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    model.eval()
    loader = DataLoader(dataset, batch_size=16)
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            probs = torch.sigmoid(model(batch.x, batch.edge_index, batch.batch).squeeze())
            preds.extend(probs.tolist())
            labels.extend(batch.y.squeeze().tolist())
    binary = [1 if p > 0.5 else 0 for p in preds]
    return {
        'accuracy': accuracy_score(labels, binary),
        'f1':       f1_score(labels, binary, zero_division=0),
        'auc':      roc_auc_score(labels, preds)
    }, preds


def save_model(model, path):
    torch.save(model.state_dict(), path)
    print(f'Saved to {path}')

def load_model(path):
    m = FakeNewsGAT()
    m.load_state_dict(torch.load(path, weights_only=True))
    m.eval()
    return m

## Step 10: Dataset Build

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

df = pd.read_csv(os.path.abspath('data/fake_news_dataset.csv'))
df['label_binary'] = (df['label'] == 'fake').astype(int)
print(f"Dataset: {len(df)} | Balance: {df['label_binary'].value_counts().to_dict()}")


def build_dataset(df, text_col='text', label_col='label_binary', sample=None):
    if sample:
        df = df.sample(sample, random_state=42)

    pyg_data, all_scored = [], []
    n_aug, n_fall = 0, 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Building graphs'):
        text, label = row[text_col], int(row[label_col])
        facts = extract_facts(text)
        if not facts:
            all_scored.append([])
            continue

        G = build_knowledge_graph(facts, use_nli=True)

        scored_docs = []
        try:
            docs = collect_evidence(text, k=10)
            if docs:
                scored_docs = score_documents_dbscan(docs)
                G           = augment_knowledge_graph(G, scored_docs, use_nli=True)
                n_aug += 1
            else:
                n_fall += 1
        except Exception as e:
            print(f'  Evidence failed: {e}')
            n_fall += 1

        pyg_data.append(graph_to_pyg(G, label=label))
        all_scored.append(scored_docs)

    print(f'Built {len(pyg_data)} | Augmented: {n_aug} | Fallback: {n_fall}')
    return pyg_data, all_scored


pyg_dataset, all_scored_docs = build_dataset(df, sample=200)

Dataset: 20000 | Balance: {1: 10056, 0: 9944}


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]C:\Users\bhada\AppData\Local\Temp\ipykernel_42056\796951813.py:21: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:   0%|          | 1/200 [00:00<01:28,  2.25it/s]

  Search failed for "better fall meet measure": [Errno 2] No such file or directory: 'search_cache\\facc9eb4cb91a77b34c78e183e875008.json'


C:\Users\bhada\AppData\Local\Temp\ipykernel_42056\796951813.py:21: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:   1%|          | 2/200 [00:00<01:10,  2.83it/s]

  Search failed for "Mr prevent anyone": [Errno 2] No such file or directory: 'search_cache\\6dad2a3c7653062675e8e4db9f775243.json'


...

  Search failed for "light tend sing": [Errno 2] No such file or directory: 'search_cache\\0af1d748245fffc6ea3a635258f316c1.json'


C:\Users\bhada\AppData\Local\Temp\ipykernel_42056\796951813.py:21: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs: 100%|█████████▉| 199/200 [02:53<00:01,  1.24s/it]

  Search failed for "camera wear understand cold attention return": [Errno 2] No such file or directory: 'search_cache\\9dfa8d146285a6478795cdb83c6bd22d.json'


C:\Users\bhada\AppData\Local\Temp\ipykernel_42056\796951813.py:21: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs: 100%|██████████| 200/200 [02:54<00:00,  1.15it/s]

  Search failed for "chance decide consumer sell writer brother method mean serve": [Errno 2] No such file or directory: 'search_cache\\b61ab9c376f39883664f3cfdd47c7d6d.json'
Built 200 | Augmented: 0 | Fallback: 200


## Step 11: Train, Evaluate, Update Credibility DB

In [15]:
indices                  = list(range(len(pyg_dataset)))
train_idx, temp_idx      = train_test_split(indices, test_size=0.3, random_state=42)
val_idx, test_idx        = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_data = [pyg_dataset[i] for i in train_idx]
val_data   = [pyg_dataset[i] for i in val_idx]
test_data  = [pyg_dataset[i] for i in test_idx]
print(f'Split: {len(train_data)} train | {len(val_data)} val | {len(test_data)} test')

model = train_gat(train_data, val_data, epochs=150, patience=20)

metrics, test_probs = evaluate_model(model, test_data)
print(f'\nTest Results:')
print(f'  Accuracy: {metrics["accuracy"]:.3f}')
print(f'  F1 Score: {metrics["f1"]:.3f}')
print(f'  AUC-ROC:  {metrics["auc"]:.3f}')

# Update credibility DB from ALL splits (train+val+test)
# Using all splits maximises the signal going into the DB.
# Note: this is fine because the DB is not used to make training decisions —
# it only affects future inference runs.
model.eval()
n_db_updates = 0
all_loader   = DataLoader(pyg_dataset, batch_size=1)
with torch.no_grad():
    for i, (batch, scored_docs) in enumerate(zip(all_loader, all_scored_docs)):
        if not scored_docs:
            continue
        logit      = model(batch.x, batch.edge_index, batch.batch).squeeze()
        prob       = torch.sigmoid(logit).item()
        confidence = abs(prob - 0.5) * 2  # 0=uncertain, 1=very confident
        n_db_updates += bulk_update_from_prediction(
            scored_docs, prob, confidence, confidence_threshold=0.1
        )

print(f'\nCredibility DB updated for {n_db_updates} source entries.')
save_model(model, 'fake_news_gat.pt')

c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params

Split: 140 train | 30 val | 30 test
Epoch  10  train=0.7343  val=0.7130  lr=4.95e-04
Epoch  20  train=0.7139  val=0.6946  lr=4.78e-04
Early stopping at epoch 21

Test Results:
  Accuracy: 0.467
  F1 Score: 0.636
  AUC-ROC:  0.500

Credibility DB updated for 0 source entries.
Saved to fake_news_gat.pt


In [16]:
# Inspect credibility DB
with get_db() as conn:
    rows = conn.execute(
        'SELECT domain, alpha, beta, model_updates, user_updates '
        'FROM sources ORDER BY model_updates + user_updates DESC LIMIT 20'
    ).fetchall()

print(f'{"Domain":<40} {"Score":>6} {"Signals":>8} {"Model":>7} {"User":>6}')
print('-' * 70)
for domain, alpha, beta, m_upd, u_upd in rows:
    score = alpha / (alpha + beta)
    n     = int(alpha + beta - 4)
    print(f'{domain:<40} {score:>6.3f} {n:>8} {m_upd:>7} {u_upd:>6}')

Domain                                    Score  Signals   Model   User
----------------------------------------------------------------------
